# Chapter 7: A2A Protocol Fundamentals
## Agent Cards, Tasks, and Communication Patterns - Code Examples

This notebook contains the runnable code from Chapter 7. It builds and validates an Agent Card, discovers an agent three ways, and then talks to a live A2A agent with every pattern the chapter describes:
- The Agent Card and skill structures of the 1.0 specification, validated with the official SDK's parser
- Discovery through the well-known endpoint, a curated registry, and direct evaluation of candidates
- The Task, Message, and Part objects as they come back from a real server
- Request and response with polling, streaming over Server-Sent Events, and push notifications to a webhook
- Multi-turn conversations with a shared contextId and a task that pauses for more input
- An analyst agent backed by a language model, served over A2A and called with the chapter's client

### Setup

The dependencies for every chapter are declared in `pyproject.toml` at the repository root. From the root, run:

```bash
uv sync --all-groups
```

Then start Jupyter with `uv run jupyter lab` and select the **Agentic AI Handbook (Python 3.13)** kernel.

The last part calls the OpenAI API. Copy `.env.example` to `.env` at the repository root and add your `OPENAI_API_KEY` before running the cells.

### How the agent runs

Chapter 7 is about the client side of A2A, but a client needs an agent to talk to. Part 2 writes a small A2A agent to a file with a `%%writefile` cell and starts it as a background process on localhost. The agent is built with the official `a2a-sdk` (version 1.1.2), whose server side Chapter 8 explains in detail; here it is a fixture that returns fixed numbers so the outputs are stable. Part 6 starts a second agent whose analysis comes from `gpt-5.6-luna`. The stop cell at the end ends both processes. Everything is written inside a temporary workspace created in the setup cell, so nothing in this repository is modified.

The chapter's listings use example domains such as `api.fintech-analytics.example.com`. This notebook points them at the local agents instead. Every listing is otherwise reproduced as printed.

In [ ]:
# Import required libraries
import os
import sys
import json
import time
import uuid
import socket
import logging
import threading
import tempfile
import subprocess
from pathlib import Path
from typing import Any, Dict, Iterator, List
from dotenv import load_dotenv

import requests

# Load environment variables (API keys)
load_dotenv()

# Verify API keys are loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment"

# Every server file lives in a throwaway workspace
WORKSPACE = Path(tempfile.mkdtemp(prefix="ch7-a2a-")).resolve()
os.chdir(WORKSPACE)

# Servers are launched with the same interpreter that runs this notebook
PYTHON = sys.executable


def wait_for_port(port: int, timeout: float = 10.0) -> None:
    """Block until a local port accepts connections."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            socket.create_connection(("127.0.0.1", port), timeout=0.2).close()
            return
        except OSError:
            time.sleep(0.2)
    raise RuntimeError(f"nothing is listening on port {port}")


print("Environment setup complete")
print(f"Workspace: {WORKSPACE}")

## Part 1: Agent Cards

An Agent Card is the JSON document an agent publishes so that others can discover it. The listing below is the chapter's card for a financial analyst agent. Rather than trusting the shape by eye, the cell parses it with the official SDK, whose `AgentCard` type is generated from the specification's normative `a2a.proto`. A field the specification does not define would make `ParseDict` raise.

In [ ]:
from google.protobuf import json_format
from a2a.types import AgentCard, AgentSkill, Part

AGENT_CARD = {
  "name": "FinancialAnalystAgent",
  "description": "Analyzes financial statements and calculates liquidity and solvency ratios",
  "version": "1.2.0",
  "provider": {
    "organization": "Fintech Analytics",
    "url": "https://fintech-analytics.example.com"
  },
  "supportedInterfaces": [
    {
      "url": "https://api.fintech-analytics.example.com/a2a/v1",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "capabilities": {"streaming": True, "pushNotifications": True},
  "securitySchemes": {
    "oauth": {
      "oauth2SecurityScheme": {
        "flows": {
          "clientCredentials": {
            "tokenUrl": "https://auth.fintech-analytics.example.com/token",
            "scopes": {
              "financial.read": "Read financial statements",
              "financial.analyze": "Run analyses"
            }
          }
        }
      }
    }
  },
  "securityRequirements": [
    {"schemes": {"oauth": {"list": ["financial.read", "financial.analyze"]}}}
  ],
  "defaultInputModes": ["text/plain", "application/pdf"],
  "defaultOutputModes": ["text/plain", "application/json"],
  "skills": [
    {
      "id": "analyze_balance_sheet",
      "name": "Analyze balance sheet",
      "description": "Analyzes a balance sheet to calculate liquidity and solvency ratios",
      "tags": ["finance", "ratios"],
      "examples": ["Calculate the current ratio from the attached balance sheet"]
    }
  ]
}

# --- Run it ---
card = json_format.ParseDict(AGENT_CARD, AgentCard())
print("=== Agent Card parsed by the SDK ===")
print(f"name:        {card.name} (version {card.version})")
print(f"interface:   {card.supported_interfaces[0].protocol_binding} {card.supported_interfaces[0].protocol_version} at {card.supported_interfaces[0].url}")
print(f"streaming:   {card.capabilities.streaming}, push notifications: {card.capabilities.push_notifications}")
print(f"security:    {list(card.security_schemes.keys())} required: {list(card.security_requirements[0].schemes['oauth'].list)}")
print(f"skills:      {[s.id for s in card.skills]}")

# A field outside the specification is rejected
try:
    json_format.ParseDict({**AGENT_CARD, "authentication": {"type": "oauth2"}}, AgentCard())
except json_format.ParseError as exc:
    print(f"\nrejected: {exc}")

### Skills

A skill in 1.0 has no input schema. Its `description` states what it accepts and returns, its `examples` show a calling agent which requests fit, and its `inputModes` and `outputModes` name the media types it handles. The chapter's sentiment skill shows the pattern.

In [ ]:
SENTIMENT_SKILL = {
  "id": "analyze_sentiment",
  "name": "Analyze sentiment",
  "description": "Scores the sentiment of a text across polarity, emotions, sarcasm, and intensity. Accepts a single sentence or a multi-paragraph document of up to 50,000 characters in any language identified by an ISO 639-1 code. Returns simple scores by default, or scores with explanations when the request asks for detail.",
  "tags": ["nlp", "sentiment", "text-analysis"],
  "examples": [
    "What is the overall sentiment of this product review?",
    "Score the emotions in this paragraph and explain each score",
    "Is this tweet sarcastic?"
  ],
  "inputModes": ["text/plain"],
  "outputModes": ["application/json", "text/plain"]
}

# --- Run it ---
skill = json_format.ParseDict(SENTIMENT_SKILL, AgentSkill())
print(f"skill {skill.id}: {len(skill.examples)} examples, tags {list(skill.tags)}")
print(f"accepts {list(skill.input_modes)}, returns {list(skill.output_modes)}")

## Part 2: Discovery

### The agent

The file below is a complete A2A agent built with the SDK: an executor that does the work, the Agent Card from Part 1 with its URL pointed at localhost, the SDK's request handler and task store, and the routes that serve the card and the JSON-RPC binding. It also exposes a tiny registry search endpoint so the chapter's registry client has something to query. Chapter 8 walks through every one of these pieces; for now, treat it as the agent on the other end of the wire.

Two behaviors are worth knowing. A balance sheet request is answered in three streamed chunks plus a structured ratios artifact, and a revenue projection request pauses in the input-required state until it is told a growth rate.

In [ ]:
%%writefile financial_agent.py
"""A small A2A 1.0 agent built with the official a2a-sdk (server building is covered in Chapter 8)."""
import asyncio
import re
import sys

import httpx
import uvicorn
from google.protobuf import json_format
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route

from a2a.helpers import (get_message_text, new_data_artifact_update_event, new_task_from_user_message,
                         new_text_artifact_update_event, new_text_message, new_text_status_update_event)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import BasePushNotificationSender, InMemoryPushNotificationConfigStore, InMemoryTaskStore
from a2a.types import AgentCard, TaskState

PORT = int(sys.argv[1]) if len(sys.argv) > 1 else 8765
BASE_URL = f"http://127.0.0.1:{PORT}"

CARD = {
    "name": "FinancialAnalystAgent",
    "description": "Analyzes financial statements and calculates liquidity and solvency ratios",
    "version": "1.2.0",
    "provider": {"organization": "Fintech Analytics", "url": "https://fintech-analytics.example.com"},
    "supportedInterfaces": [
        {"url": f"{BASE_URL}/a2a/v1", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"}
    ],
    "capabilities": {"streaming": True, "pushNotifications": True},
    "securitySchemes": {"bearer": {"httpAuthSecurityScheme": {"scheme": "bearer"}}},
    "securityRequirements": [{"schemes": {"bearer": {"list": []}}}],
    "defaultInputModes": ["text/plain", "application/pdf"],
    "defaultOutputModes": ["text/plain", "application/json"],
    "skills": [
        {
            "id": "analyze_balance_sheet",
            "name": "Analyze balance sheet",
            "description": "Analyzes a balance sheet to calculate liquidity and solvency ratios",
            "tags": ["finance", "ratios"],
            "examples": ["Calculate the current ratio from the attached balance sheet"],
        },
        {
            "id": "project_revenue",
            "name": "Project revenue",
            "description": "Projects next year's revenue from a growth assumption; asks for the rate if it is missing",
            "tags": ["finance", "forecast"],
            "examples": ["Project next year's revenue assuming 8 percent growth"],
        },
    ],
}

RATIOS = {"currentRatio": 1.52, "quickRatio": 1.18, "cashRatio": 0.45}


class FinancialAnalystExecutor(AgentExecutor):
    """Deterministic analyst: the numbers are fixed so the notebook output is stable."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        text = context.get_user_input()
        task = context.current_task or new_task_from_user_message(context.message)
        if context.current_task is None:
            await event_queue.enqueue_event(task)  # a new Task always goes first
        # A resumed task keeps its history, so decide the skill from the whole conversation
        conversation = " ".join(get_message_text(m) for m in task.history) + " " + text
        await asyncio.sleep(0.2)
        await event_queue.enqueue_event(new_text_status_update_event(
            task.id, task.context_id, TaskState.TASK_STATE_WORKING, "Analyzing"))

        if "project" in conversation.lower():
            rate = re.search(r"(\d+(?:\.\d+)?)\s*(?:percent|%)", text.lower())
            if not rate:
                await event_queue.enqueue_event(new_text_status_update_event(
                    task.id, task.context_id, TaskState.TASK_STATE_INPUT_REQUIRED,
                    "Which growth rate should I assume?"))
                return
            growth = float(rate.group(1))
            projected = round(48.0 * (1 + growth / 100), 2)
            await event_queue.enqueue_event(new_data_artifact_update_event(
                task.id, task.context_id, "projection",
                {"baseRevenueMillions": 48.0, "growthPercent": growth, "projectedRevenueMillions": projected},
                media_type="application/json"))
            await event_queue.enqueue_event(new_text_status_update_event(
                task.id, task.context_id, TaskState.TASK_STATE_COMPLETED, "Done"))
            return

        # Balance sheet analysis, streamed as three chunks so SSE clients see progress
        chunks = ["Acme Corp current ratio 1.52, ", "quick ratio 1.18, ", "cash ratio 0.45. Liquidity looks healthy."]
        artifact_id = None
        for i, chunk in enumerate(chunks):
            event = new_text_artifact_update_event(
                task.id, task.context_id, "analysis", chunk,
                append=i > 0, last_chunk=i == len(chunks) - 1, artifact_id=artifact_id)
            artifact_id = event.artifact.artifact_id
            await event_queue.enqueue_event(event)
            await asyncio.sleep(0.15)
        await event_queue.enqueue_event(new_data_artifact_update_event(
            task.id, task.context_id, "ratios", {"ratios": RATIOS, "assessment": "healthy", "confidence": 0.92},
            media_type="application/json"))
        await event_queue.enqueue_event(new_text_status_update_event(
            task.id, task.context_id, TaskState.TASK_STATE_COMPLETED, "Done"))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(new_text_status_update_event(
            context.task_id, context.context_id, TaskState.TASK_STATE_CANCELED, "Canceled"))


card = json_format.ParseDict(CARD, AgentCard())
push_store = InMemoryPushNotificationConfigStore()
handler = DefaultRequestHandler(
    agent_executor=FinancialAnalystExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=card,
    push_config_store=push_store,
    push_sender=BasePushNotificationSender(httpx.AsyncClient(timeout=10), push_store),
)


async def search_agents(request: Request):
    """A tiny curated registry: filter this server's card by skill id and tag."""
    wanted_skills = {s for s in request.query_params.get("skills", "").split(",") if s}
    wanted_tags = {t for t in request.query_params.get("tags", "").split(",") if t}
    have_skills = {s["id"] for s in CARD["skills"]}
    have_tags = {t for s in CARD["skills"] for t in s["tags"]}
    match = wanted_skills <= have_skills and wanted_tags <= have_tags
    return JSONResponse({"agents": [CARD] if match else []})


app = Starlette(routes=[
    *create_agent_card_routes(card),
    *create_jsonrpc_routes(handler, rpc_url="/a2a/v1"),
    Route("/agents/search", search_agents, methods=["GET"]),
])

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

### Starting the agent

This cell starts the agent as a background process and waits until its port accepts connections. It keeps running until the stop cell at the end of the notebook.

In [ ]:
AGENT_PORT = 8765
AGENT_BASE = f"http://127.0.0.1:{AGENT_PORT}"

agent_process = subprocess.Popen(
    [PYTHON, "financial_agent.py", str(AGENT_PORT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_port(AGENT_PORT)
print(f"=== FinancialAnalystAgent running ===")
print(f"pid {agent_process.pid}, base URL {AGENT_BASE}")

### Well-known endpoint

The simplest discovery method is a GET on `/.well-known/agent-card.json` under the agent's base URL. The chapter's helper builds that URL and returns the parsed card.

In [ ]:
def fetch_agent_card(base_url: str) -> Dict[str, Any]:
    """Fetch agent card from well-known endpoint."""
    url = f"{base_url}/.well-known/agent-card.json"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()


# --- Run it ---
live_card = fetch_agent_card(AGENT_BASE)
print("=== Card from the well-known endpoint ===")
print(f"name:      {live_card['name']}")
print(f"interface: {live_card['supportedInterfaces'][0]['url']}")
print(f"skills:    {[s['id'] for s in live_card['skills']]}")

AGENT_URL = live_card["supportedInterfaces"][0]["url"]

### Curated registry

In an enterprise or a marketplace, a registry holds many cards and answers searches. The chapter shows only the `search_agents` method; the class below adds the constructor it implies. The local agent serves a minimal `/agents/search` endpoint that filters its own card by skill id and tag, which is enough to exercise the client.

In [ ]:
class AgentRegistry:
    """Client for a curated registry of Agent Cards."""

    def __init__(self, registry_url: str, api_key: str):
        self.registry_url = registry_url.rstrip("/")
        self.headers = {"Authorization": f"Bearer {api_key}"}

    def search_agents(self, skills: List[str], tags: List[str]) -> List[Dict]:
        """Search for agents matching criteria."""
        params = {'skills': ','.join(skills), 'tags': ','.join(tags)}
        response = requests.get(f"{self.registry_url}/agents/search", params=params,
                                headers=self.headers, timeout=10)
        response.raise_for_status()
        return response.json()['agents']


# --- Run it ---
registry = AgentRegistry(AGENT_BASE, "api-key")
agents = registry.search_agents(
    skills=["analyze_balance_sheet"],
    tags=["finance"]
)
print("=== Registry search ===")
print(f"finance agents with analyze_balance_sheet: {[a['name'] for a in agents]}")
print(f"translation agents: {registry.search_agents(skills=['translate_document'], tags=[])}")

### Evaluating candidates

Discovery produces candidates; the client still has to pick one. The chapter's evaluator matches skill ids and then capability flags. The decoy card below advertises the skill but not streaming, so it drops out when streaming is required.

In [ ]:
def evaluate_agents(candidates, required_skills, required_caps=None):
    """Select best agent from candidates."""
    # Filter by skills
    matches = [a for a in candidates
               if all(skill in {s['id'] for s in a['skills']}
                     for skill in required_skills)]

    # Filter by capabilities if specified
    if required_caps:
        matches = [a for a in matches
                   if all(a['capabilities'].get(k) == v
                         for k, v in required_caps.items())]

    return matches[0] if matches else None


# --- Run it ---
decoy = {**live_card, "name": "BatchOnlyAnalyst", "capabilities": {"streaming": False}}
chosen = evaluate_agents([decoy, live_card], ["analyze_balance_sheet"], {"streaming": True})
print(f"needs streaming:      {chosen['name']}")
chosen = evaluate_agents([decoy, live_card], ["analyze_balance_sheet"])
print(f"any capabilities:     {chosen['name']}")
print(f"unknown skill:        {evaluate_agents([decoy, live_card], ['translate_document'])}")

## Part 3: Tasks, Messages, and Parts

### The client

The chapter's client speaks the JSON-RPC binding directly: every call is a JSON-RPC 2.0 envelope posted to the agent's interface URL with an `A2A-Version` header. `send_message` builds a message with a fresh id and the role `ROLE_USER`, `get_task` fetches a task by id, and `wait_for_completion` polls until the task reaches a terminal state.

In [ ]:
TERMINAL = {"TASK_STATE_COMPLETED", "TASK_STATE_FAILED",
            "TASK_STATE_CANCELED", "TASK_STATE_REJECTED"}


class A2AClient:
    """Minimal A2A client for the JSON-RPC binding."""

    def __init__(self, agent_url: str, token: str):
        self.agent_url = agent_url
        self.headers = {"A2A-Version": "1.0", "Authorization": f"Bearer {token}"}

    def call(self, method: str, params: dict) -> dict:
        payload = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params}
        response = requests.post(self.agent_url, json=payload,
                                 headers=self.headers, timeout=30)
        response.raise_for_status()
        body = response.json()
        if "error" in body:
            raise RuntimeError(f"{method} failed: {body['error']}")
        return body["result"]

    def send_message(self, parts: list, context_id: str | None = None,
                     task_id: str | None = None,
                     configuration: dict | None = None) -> dict:
        message = {"messageId": str(uuid.uuid4()), "role": "ROLE_USER", "parts": parts}
        if context_id:
            message["contextId"] = context_id
        if task_id:
            message["taskId"] = task_id
        params = {"message": message}
        if configuration:
            params["configuration"] = configuration
        return self.call("SendMessage", params)

    def get_task(self, task_id: str) -> dict:
        return self.call("GetTask", {"id": task_id})

    def wait_for_completion(self, task_id: str, timeout: float = 300) -> dict:
        deadline = time.time() + timeout
        while time.time() < deadline:
            task = self.get_task(task_id)
            if task["status"]["state"] in TERMINAL:
                return task
            time.sleep(2)
        raise TimeoutError(f"Task {task_id} did not finish in time")


# --- Run it ---
client = A2AClient(AGENT_URL, token="demo-token")
result = client.send_message([{"text": "Calculate the current ratio from the attached balance sheet"}])
task = result["task"]
print("=== The Task object as the server returned it ===")
print(f"id:        {task['id']}")
print(f"contextId: {task['contextId']}")
print(f"status:    {task['status']['state']}")
print(f"history:   {len(task.get('history', []))} message(s), first role {task['history'][0]['role']}")
print(f"artifacts: {[(a['name'], list(a['parts'][0].keys())) for a in task.get('artifacts', [])]}")

The SDK's request handler answered with the finished task, because by default `SendMessage` blocks until the executor reaches a terminal state. A task that takes longer, or an agent that returns immediately, comes back as `TASK_STATE_SUBMITTED` or `TASK_STATE_WORKING` instead, which is what the polling loop in Part 4 is for.

### Parts

A part carries exactly one content field: `text`, `raw`, `url`, or `data`, plus optional `filename`, `mediaType`, and `metadata`. The cell parses the chapter's three examples and asks the SDK which content field each one set.

In [ ]:
TEXT_PART = {
  "text": "Please analyze this financial statement and identify any concerning trends."
}

FILE_PART = {
  "url": "https://storage.example.com/files/quarterly_report.pdf",
  "filename": "quarterly_report.pdf",
  "mediaType": "application/pdf"
}

DATA_PART = {
  "data": {
    "ratios": {
      "currentRatio": 1.52,
      "quickRatio": 1.18,
      "cashRatio": 0.45
    },
    "assessment": "healthy",
    "confidence": 0.92
  },
  "mediaType": "application/json"
}

# --- Run it ---
for label, raw in [("text part", TEXT_PART), ("file part", FILE_PART), ("data part", DATA_PART)]:
    part = json_format.ParseDict(raw, Part())
    print(f"{label:10s} content field = {part.WhichOneof('content')!r:8s} mediaType = {part.media_type or '-'}")

# The data artifact the agent produced in Part 3 is a data part
ratios_part = json_format.ParseDict(task["artifacts"][1]["parts"][0], Part())
print(f"\nagent's ratios artifact: {json_format.MessageToDict(ratios_part)['data']}")

## Part 4: Communication Patterns

### Request and response with polling

The chapter's usage block sends one message, waits for the task, and prints each artifact's text. Against this agent the wait ends on the first poll, but the loop is the same one a client needs for an agent that answers before the work is done.

In [ ]:
result = client.send_message([{"text": "Calculate the current ratio from the attached balance sheet"}])
task = client.wait_for_completion(result["task"]["id"])
print("=== Polling result ===")
for artifact in task["artifacts"]:
    print("".join(part.get("text", "") for part in artifact["parts"]) or artifact["parts"][0].get("data"))

### Streaming with Server-Sent Events

`stream_message` posts the same envelope with the method `SendStreamingMessage` and an `Accept` header of `text/event-stream`. Each SSE data line is a JSON-RPC response whose `result` holds a task, a status update, an artifact update, or a message. The agent streams its analysis in three chunks, so the artifact text appears as it is produced.

In [ ]:
def stream_message(client: A2AClient, parts: list) -> Iterator[dict]:
    """Send a message with SendStreamingMessage and yield each event's result."""
    message = {"messageId": str(uuid.uuid4()), "role": "ROLE_USER", "parts": parts}
    payload = {"jsonrpc": "2.0", "id": 1, "method": "SendStreamingMessage",
               "params": {"message": message}}
    headers = {**client.headers, "Accept": "text/event-stream"}
    with requests.post(client.agent_url, json=payload, headers=headers,
                       stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line.startswith(b"data: "):
                yield json.loads(line[6:])["result"]


# --- Run it ---
print("=== Streaming ===")
for event in stream_message(client, [{"text": "Analyze Acme Corp financials"}]):
    if "task" in event:
        print("task", event["task"]["id"], event["task"]["status"]["state"])
    elif "artifactUpdate" in event:
        for part in event["artifactUpdate"]["artifact"]["parts"]:
            print(part.get("text", "") or f"[data part: {part.get('data')}]", end="", flush=True)
    elif "statusUpdate" in event:
        print("\nstatus", event["statusUpdate"]["status"]["state"])

### Push notifications to a webhook

For long-running work the client registers a webhook instead of holding a connection. The Flask app below receives each event by HTTP POST, checks the secret, and dispatches it to a callback by task id. The client passes the webhook URL and the secret in the request's `taskPushNotificationConfig`, with `returnImmediately` so the call returns as soon as the task exists and the events arrive afterwards.

The secret is sent twice because implementations differ. The specification says the agent must send the `authentication` credentials in the Authorization header; `a2a-sdk` 1.1.2 instead echoes the `token` field in an `X-A2A-Notification-Token` header. The receiver accepts either.

In [ ]:
# The Flask development server runs in a thread; silence its startup banner and request log
import flask.cli
flask.cli.show_server_banner = lambda *args, **kwargs: None
logging.getLogger("werkzeug").setLevel(logging.ERROR)

from flask import Flask, abort, jsonify, request

app = Flask(__name__)
WEBHOOK_SECRET = "replace-with-a-random-secret"
task_callbacks = {}


@app.post("/webhooks/a2a")
def handle_a2a_webhook():
    """Receive A2A task updates. The body holds one event, as in a stream."""
    presented = request.headers.get("X-A2A-Notification-Token") or \
        request.headers.get("Authorization", "").removeprefix("Bearer ")
    if presented != WEBHOOK_SECRET:
        abort(401)
    event = request.get_json()
    update = event.get("statusUpdate") or event.get("artifactUpdate") or {}
    task_id = update.get("taskId") or event.get("task", {}).get("id")
    if task_id in task_callbacks:
        task_callbacks[task_id](event)
    return jsonify({"status": "received"}), 200


def send_message_with_webhook(client: A2AClient, parts: list,
                              webhook_url: str) -> str:
    """Send a message and ask the agent to POST task updates to webhook_url."""
    configuration = {
        "returnImmediately": True,
        "taskPushNotificationConfig": {
            "url": webhook_url,
            "token": WEBHOOK_SECRET,
            "authentication": {"scheme": "Bearer", "credentials": WEBHOOK_SECRET},
        }
    }
    result = client.send_message(parts, configuration=configuration)
    return result["task"]["id"]


# --- Run it ---
WEBHOOK_PORT = 8767
threading.Thread(
    target=lambda: app.run(host="127.0.0.1", port=WEBHOOK_PORT, use_reloader=False),
    daemon=True,
).start()
wait_for_port(WEBHOOK_PORT)

received = []
task_id = send_message_with_webhook(
    client,
    [{"text": "Analyze quarterly trends"}],
    f"http://127.0.0.1:{WEBHOOK_PORT}/webhooks/a2a",
)
task_callbacks[task_id] = lambda event: received.append(next(iter(event)))
print(f"task {task_id} registered, waiting for notifications...")
time.sleep(2)
print(f"events delivered to the webhook: {received}")
print(f"final state by GetTask: {client.get_task(task_id)['status']['state']}")

## Part 5: Multi-Turn Conversations

The server generates a `contextId` when it creates a task and returns it in the task. Later messages that carry the same `contextId` join the conversation, and a message that also carries a `taskId` continues one specific task. The third turn asks for a projection without a growth rate, so the agent parks the task in `TASK_STATE_INPUT_REQUIRED` with its question in the status message, and the client answers by sending a message with the same task and context ids.

In [ ]:
# First turn
first = client.send_message([{"text": "What were Acme Corp's revenues in Q4 2025?"}])
task1 = client.wait_for_completion(first["task"]["id"])
context_id = task1["contextId"]

# Second turn in the same conversation
second = client.send_message(
    [{"text": "How does that compare to Q4 2024?"}],
    context_id=context_id,
)
task2 = client.wait_for_completion(second["task"]["id"])

# A task that pauses until the client supplies more input
third = client.send_message([{"text": "Project next year's revenue"}],
                            context_id=context_id)
task3 = third["task"]
if task3["status"]["state"] == "TASK_STATE_INPUT_REQUIRED":
    print("Agent asks:", task3["status"]["message"]["parts"][0]["text"])
    reply = client.send_message(
        [{"text": "Assume 8 percent growth"}],
        context_id=context_id,
        task_id=task3["id"],
    )
    task3 = client.wait_for_completion(reply["task"]["id"])

# --- Inspect ---
print("\n=== Conversation ===")
print(f"context shared by all three tasks: {task1['contextId'] == task2['contextId'] == task3['contextId']}")
print(f"task 3 final state: {task3['status']['state']}")
print(f"task 3 history: {[m['role'] + ': ' + m['parts'][0]['text'][:40] for m in task3['history'] if m['parts'][0].get('text')]}")
print(f"projection: {task3['artifacts'][0]['parts'][0]['data']}")

## Part 6: An Analyst Agent Backed by a Language Model

Nothing in the protocol changes when the work behind the executor is a model call. The agent below has the same card shape as the fixture, but its executor sends the balance sheet to `gpt-5.6-luna` on the OpenAI Responses API and returns the model's analysis as a text artifact. The client is the chapter's `A2AClient`, unchanged.

In [ ]:
%%writefile llm_analyst_agent.py
import sys

import uvicorn
from google.protobuf import json_format
from openai import AsyncOpenAI
from starlette.applications import Starlette

from a2a.helpers import (new_task_from_user_message, new_text_artifact_update_event,
                         new_text_status_update_event)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, TaskState

PORT = int(sys.argv[1]) if len(sys.argv) > 1 else 8766
MODEL = "gpt-5.6-luna"

CARD = {
    "name": "LLMFinancialAnalyst",
    "description": "Reads a balance sheet summary and explains its liquidity position",
    "version": "1.0.0",
    "supportedInterfaces": [
        {"url": f"http://127.0.0.1:{PORT}/a2a/v1", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"}
    ],
    "capabilities": {"streaming": True},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [{
        "id": "analyze_balance_sheet",
        "name": "Analyze balance sheet",
        "description": "Computes liquidity ratios from the figures in the request and comments on them in three sentences",
        "tags": ["finance", "ratios"],
        "examples": ["Current assets 12.5M, current liabilities 8.2M, inventory 2.8M, cash 3.7M. Assess liquidity."],
    }],
}

INSTRUCTIONS = (
    "You are a financial analyst. From the balance sheet figures in the request, compute the current ratio, "
    "quick ratio, and cash ratio, state each to two decimals, and assess liquidity in three sentences."
)


class LLMAnalystExecutor(AgentExecutor):
    def __init__(self):
        self.openai = AsyncOpenAI()

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task or new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)
        await event_queue.enqueue_event(new_text_status_update_event(
            task.id, task.context_id, TaskState.TASK_STATE_WORKING, "Calling the model"))
        try:
            response = await self.openai.responses.create(
                model=MODEL, instructions=INSTRUCTIONS, input=context.get_user_input())
            await event_queue.enqueue_event(new_text_artifact_update_event(
                task.id, task.context_id, "analysis", response.output_text))
            await event_queue.enqueue_event(new_text_status_update_event(
                task.id, task.context_id, TaskState.TASK_STATE_COMPLETED, "Done"))
        except Exception as exc:
            await event_queue.enqueue_event(new_text_status_update_event(
                task.id, task.context_id, TaskState.TASK_STATE_FAILED, str(exc)))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(new_text_status_update_event(
            context.task_id, context.context_id, TaskState.TASK_STATE_CANCELED, "Canceled"))


card = json_format.ParseDict(CARD, AgentCard())
handler = DefaultRequestHandler(agent_executor=LLMAnalystExecutor(), task_store=InMemoryTaskStore(), agent_card=card)
app = Starlette(routes=[*create_agent_card_routes(card), *create_jsonrpc_routes(handler, rpc_url="/a2a/v1")])

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

In [ ]:
LLM_PORT = 8766
llm_process = subprocess.Popen(
    [PYTHON, "llm_analyst_agent.py", str(LLM_PORT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_port(LLM_PORT)

llm_card = fetch_agent_card(f"http://127.0.0.1:{LLM_PORT}")
llm_client = A2AClient(llm_card["supportedInterfaces"][0]["url"], token="demo-token")

balance_sheet = (
    "Acme Corp balance sheet, Q4 2025: current assets 12.5M, of which inventory 2.8M and cash 3.7M; "
    "current liabilities 8.2M; long-term debt 15.0M; total equity 22.4M. Assess liquidity."
)
result = llm_client.send_message([{"text": balance_sheet}])
task = llm_client.wait_for_completion(result["task"]["id"])

print(f"=== {llm_card['name']} ({task['status']['state']}) ===")
for artifact in task["artifacts"]:
    print(artifact["parts"][0]["text"])

### Stopping the agents

The client closed its own connections when each call returned. The two agent processes are ours to stop.

In [ ]:
for name, proc in [("FinancialAnalystAgent", agent_process), ("LLMFinancialAnalyst", llm_process)]:
    proc.terminate()
    proc.wait(timeout=5)
    print(f"{name} stopped with return code {proc.returncode}")

## Summary

In this notebook, we implemented:

1. **Agent Cards**: The chapter's card and skill, validated with the SDK's specification-derived types, including a rejected pre-1.0 field
2. **Discovery**: The well-known endpoint, a registry search, and candidate evaluation by skill id and capability flags
3. **Tasks, Messages, and Parts**: A real Task with its id, contextId, status, history, and artifacts, and the three part types
4. **Polling**: The chapter's JSON-RPC client sending `SendMessage` and looping on `GetTask` until a terminal state
5. **Streaming**: `SendStreamingMessage` over Server-Sent Events, with artifact chunks printed as they arrived
6. **Push Notifications**: A webhook that verified the registered secret and received status events from the agent
7. **Multi-Turn**: Three tasks in one context, one of which paused for input and resumed with the same task id
8. **A Model-Backed Agent**: The same protocol with `gpt-5.6-luna` doing the analysis behind the executor

### Key Takeaways:

- The 1.0 data model is the specification's protobuf; parsing a card or a part with the SDK is a validity check
- Discovery is a lookup; the client still has to evaluate candidates against the skills and capabilities it needs
- A task's outputs live in artifacts, its conversation in history, and its state in the status object
- The three delivery patterns share one task model; only the transport of updates changes
- `contextId` groups tasks into a conversation, and `taskId` continues one task, which is how input-required works

### Next Steps:

- Replace the fixture's fixed numbers with a real data source and add a `securityRequirements` check on the server
- Put the model-backed agent behind the authentication middleware from Chapter 8
- Move on to Chapter 8, which builds A2A servers with the SDK, integrates the major frameworks, and secures the result